# Instrumental Variables (IV)

- 전제 조건, LATE (complier)
- 2SLS Estimator
- Weak IV, DML 응용 등 (causal-ml book에 다양한 상황이 제시되어 있음. 실용적인 내용은 최대한 다루기)

In [ ]:
% pip install linearmodels

In [3]:
import pandas as pd
import numpy as np
from linearmodels.iv import IV2SLS

push delivered(푸시 메세지 전달)와 In-App 구매력 사이의 연관관계는 인과관계가 될 수 없습니다. 소득이 confouder로 작용하기 때문입니다. (부유한 고객은 최신 스마트폰을 가져 푸시 메세지를 잘 받고, 동시에 In-App 구매력도 높기 때문입니다.)

IV를 사용할 때, exclusion restriction가 반드시 필요합니다. 이는 정량적으로 검증할 수 없지만, 이 경우에 대해서는 push assigned(푸시 할당)는 랜덤 할당이고 다른 채널이 없기 때문에 exclusion restriction을 쉽게 주장할 수 있습니다. 다시 말해, Push Assigned는 반드시 Push Delivered를 통해서만 구매에 영향을 미칩니다.

In [6]:
data = pd.read_csv("../data/matheus_data/app_engagement_push.csv")
data.head()

,in_app_purchase,push_assigned,push_delivered
0,47,1,1
1,43,1,0
2,51,1,1
3,49,0,0
4,79,0,0


### 1st Stage (Relevance)

In [14]:
print("1st Stage: Push Assignment -> Push Delivered")
first_stage = IV2SLS.from_formula("push_delivered ~ 1 + push_assigned", data).fit()
print(first_stage.summary.tables[1])
print(f"Compliance rate: {first_stage.params['push_assigned']:.2%}")

1st Stage: Push Assignment -> Push Delivered
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
Intercept       2.22e-16                                                         
push_assigned     0.7176     0.0064     112.07     0.0000      0.7050      0.7301
Compliance rate: 71.76%


/Users/sanakang/anaconda3/lib/python3.11/site-packages/linearmodels/iv/results.py:198: RuntimeWarning: invalid value encountered in sqrt
  std_errors = sqrt(diag(self.cov))


### 2SLS Estimation: LATE

In [17]:
iv_model = IV2SLS.from_formula("in_app_purchase ~ 1 + [push_delivered ~ push_assigned]", data).fit()
print(iv_model.summary.tables[1])

                               Parameter Estimates                                
                Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
----------------------------------------------------------------------------------
Intercept          69.292     0.3624     191.22     0.0000      68.581      70.002
push_delivered     3.2938     0.7165     4.5974     0.0000      1.8896      4.6981


In [18]:
late_estimate = iv_model.params['push_delivered']
ci_lower = late_estimate - 1.96 * iv_model.std_errors['push_delivered'] 
ci_upper = late_estimate + 1.96 * iv_model.std_errors['push_delivered']

print(f"LATE 추정치: {late_estimate:.3f}")
print(f"95% 신뢰구간: [{ci_lower:.3f}, {ci_upper:.3f}]")

LATE 추정치: 3.294
95% 신뢰구간: [1.890, 4.698]


### 질문이나 의견을 남겨주세요.
<script src="https://utteranc.es/client.js"
        repo="CausalInferenceLab/awesome-causal-inference-python"
        issue-term="pathname"
        theme="github-light"
        crossorigin="anonymous"
        async>
</script>